# 04. Trajectory Analysis (PAGA + DPT / scVelo)

**목적**: 세포 분화/발달 경로 분석

BAM 파일 또는 loom 파일 유무에 따라 자동으로 분기:
- `scvelo_ready`: scVelo RNA velocity (방향성 화살표)
- `bam_available`: velocyto로 loom 생성 후 scVelo
- `no_velocity`: PAGA + DPT + CytoTRACE

**입력**: `output/checkpoints/{dataset}/08_annotated.h5ad`  
**출력**: `output/figures/trajectory/{dataset}/`, `output/checkpoints/{dataset}/09_trajectory.h5ad`


In [ ]:
import sys
sys.path.insert(0, "../../")
sys.path.insert(0, "../")

import scanpy as sc
import pandas as pd
import numpy as np
from pathlib import Path
import yaml

from modules.io import load_adata, save_adata, detect_velocity_mode
from modules.plotting import umap_panel

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white")

In [ ]:
DATASET = "human_genes_only"
CHECKPOINT_DIR = f"../../output/checkpoints/{DATASET}"
FIG_OUT_DIR = Path(f"../../output/figures/trajectory/{DATASET}")
FIG_OUT_DIR.mkdir(parents=True, exist_ok=True)

# samples.yaml에서 velocity 경로 확인
with open("../../configs/samples.yaml") as f:
    samples_cfg = yaml.safe_load(f)

velocity_cfg = samples_cfg.get("velocity", {})
bam_dir = velocity_cfg.get("bam_dir", "") or "../../data"
loom_dir = velocity_cfg.get("loom_dir", "") or "../../data"

adata = load_adata(f"{CHECKPOINT_DIR}/08_annotated.h5ad")
print(adata)

## 0. Velocity 모드 감지

In [ ]:
velocity_mode = detect_velocity_mode("../../data")
print(f"Detected velocity mode: {velocity_mode}")
print()
print({
    "scvelo_ready": "loom or spliced layer found → running scVelo",
    "bam_available": "BAM files found → need to run velocyto first",
    "no_velocity": "no velocity data → using PAGA + DPT + CytoTRACE",
}.get(velocity_mode, "unknown"))

## 1. PAGA (모든 경로 공통)

In [ ]:
# PAGA trajectory
groupby = "cell_type" if "cell_type" in adata.obs.columns else "leiden"

sc.tl.paga(adata, groups=groupby)
sc.pl.paga(
    adata,
    color=groupby,
    title="PAGA graph",
    save=None,
)

In [ ]:
# PAGA 가이드 UMAP (edge 두께 = 연결 강도)
sc.tl.draw_graph(adata, init_pos="paga")
sc.pl.draw_graph(adata, color=groupby, legend_loc="right margin")

## 2A. scVelo (BAM/loom 있는 경우)

In [ ]:
if velocity_mode == "bam_available":
    print("BAM files found. Run velocyto first:")
    print(f"  velocyto run -b <barcodes.tsv> -o {loom_dir} <bam_file> <gtf_file>")
    print("Then set 'loom_dir' in configs/samples.yaml and re-run.")

elif velocity_mode == "scvelo_ready":
    try:
        import scvelo as scv
        scv.settings.verbosity = 2
        scv.settings.set_figure_params("scvelo")

        # loom 파일에서 spliced/unspliced 로드
        loom_files = list(Path(loom_dir).rglob("*.loom"))
        if loom_files:
            ldata = scv.read(str(loom_files[0]), cache=True)
            adata = scv.utils.merge(adata, ldata)

        scv.pp.filter_and_normalize(adata)
        scv.pp.moments(adata)
        scv.tl.velocity(adata)
        scv.tl.velocity_graph(adata)
        scv.pl.velocity_embedding_stream(
            adata,
            basis="umap",
            color=groupby,
            save=str(FIG_OUT_DIR / "velocity_stream.png"),
        )
        print("scVelo velocity analysis complete")
    except ImportError:
        print("scvelo not installed. Run: pip install scvelo")
else:
    print("No velocity data → proceeding with PAGA + DPT + CytoTRACE")

## 2B. CytoTRACE + DPT (velocity 없는 경우)

In [ ]:
if velocity_mode == "no_velocity":
    # CytoTRACE: 유전자 발현 복잡도 기반 분화 상태 추정
    try:
        import cytotrace2
        ct_result = cytotrace2.cytotrace2(adata, slot_type="counts", species="human")
        adata.obs["cytotrace_score"] = ct_result["CytoTRACE2_Score"]
        # 높은 점수 = 미분화 (stem-like)
        print("CytoTRACE2 score calculated")
        print("High score = undifferentiated (stem-like state)")
        sc.pl.umap(adata, color="cytotrace_score", color_map="RdYlBu_r", title="CytoTRACE2 Score")
    except ImportError:
        print("cytotrace2 not installed. Run: pip install cytotrace2")
        # 간단 대안: 유전자 수를 분화도 대리 지표로 사용
        adata.obs["cytotrace_proxy"] = adata.obs["n_genes_by_counts"]
        sc.pl.umap(adata, color="cytotrace_proxy", title="n_genes (CytoTRACE proxy)")

In [ ]:
if velocity_mode == "no_velocity":
    # DPT pseudotime
    # Root cell = CytoTRACE 점수가 가장 높은 세포 (가장 미분화)
    score_col = "cytotrace_score" if "cytotrace_score" in adata.obs.columns else "n_genes_by_counts"
    root_idx = adata.obs[score_col].idxmax()
    root_cell_idx = adata.obs_names.get_loc(root_idx)
    
    adata.uns["iroot"] = root_cell_idx
    sc.tl.diffmap(adata)
    sc.tl.dpt(adata)
    
    print(f"Root cell: {root_idx} (highest {score_col})")
    sc.pl.umap(adata, color="dpt_pseudotime", color_map="viridis", title="DPT Pseudotime")

## 3. Pseudotime에 따른 유전자 발현 변화

In [ ]:
# ============================================================
# 수동 설정: trajectory에 따라 변화하는 유전자 (관심 유전자 입력)
# ============================================================
TRAJECTORY_GENES = []  # 예: ["MKI67", "CD44", "CD3D", "EPCAM"]

if TRAJECTORY_GENES and "dpt_pseudotime" in adata.obs.columns:
    # 유전자 발현 vs pseudotime 플롯
    sc.pl.scatter(
        adata,
        x="dpt_pseudotime",
        y=TRAJECTORY_GENES[0],
        color=groupby,
    )
else:
    print("Set TRAJECTORY_GENES to visualize gene expression along pseudotime")

In [ ]:
# 저장
out_path = f"{CHECKPOINT_DIR}/09_trajectory.h5ad"
save_adata(adata, out_path)
print(f"Saved: {out_path}")